In [1]:
import sys
sys.path.append('../src')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from pathlib import Path
from collections import Counter
import pandas as pd

from data.labelme_loader import (
    load_all_annotations,
    load_labelme_json,
    load_image_from_labelme,
    polygon_to_mask,
    VALID_CLASSES
)

# Plot settings
plt.rcParams['figure.figsize'] = (12, 8)
sns.set_style('whitegrid')

# Data directory
DATA_DIR = Path('../data/raw')

print(f"Data directory: {DATA_DIR.absolute()}")
print(f"Valid classes: {VALID_CLASSES}")

Data directory: C:\Users\mswr\dev\facies\notebooks\..\data\raw
Valid classes: ['Peloid', 'Ooid', 'Broken ooid', 'Intraclast']


## 1. Load All Annotations

In [ ]:
# Load all annotations (filtered to valid classes only)
annotations = load_all_annotations(DATA_DIR, filter_classes=True)

print(f"Total images: {len(annotations)}")
print(f"\nImages loaded:")
for img_name in sorted(annotations.keys()):
    n_grains = len(annotations[img_name])
    print(f"  {img_name}: {n_grains} grains")

## 2. Overall Class Distribution

In [ ]:
# Count classes across all images
all_labels = []
for grains in annotations.values():
    all_labels.extend([g['label'] for g in grains])

class_counts = Counter(all_labels)
total_grains = len(all_labels)

# Create dataframe
df_classes = pd.DataFrame([
    {'Class': cls, 'Count': count, 'Percentage': f"{count/total_grains*100:.2f}%"}
    for cls, count in class_counts.most_common()
])

print(f"Total grains (filtered): {total_grains}")
print(f"\nClass distribution:")
display(df_classes)

# Bar plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
classes = [item['Class'] for item in df_classes.to_dict('records')]
counts = [item['Count'] for item in df_classes.to_dict('records')]
colors = sns.color_palette('husl', len(classes))

ax1.bar(classes, counts, color=colors)
ax1.set_ylabel('Count')
ax1.set_title('Class Distribution (Linear Scale)')
ax1.tick_params(axis='x', rotation=45)

# Log scale to show rare classes
ax2.bar(classes, counts, color=colors)
ax2.set_ylabel('Count (log scale)')
ax2.set_yscale('log')
ax2.set_title('Class Distribution (Log Scale)')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3. Per-Image Class Distribution

In [ ]:
# Build per-image class matrix
image_class_data = []

for img_name, grains in annotations.items():
    class_counts_img = Counter([g['label'] for g in grains])
    row = {'Image': img_name}
    for cls in VALID_CLASSES:
        row[cls] = class_counts_img.get(cls, 0)
    row['Total'] = len(grains)
    image_class_data.append(row)

df_images = pd.DataFrame(image_class_data)
df_images = df_images.sort_values('Total', ascending=False)

print("Per-image breakdown:")
display(df_images)

# Identify critical images (containing broken ooids)
broken_ooid_images = df_images[df_images['Broken ooid'] > 0]['Image'].tolist()
print(f"\n⚠️ Images with Broken ooids ({len(broken_ooid_images)}):")
for img in broken_ooid_images:
    n = df_images[df_images['Image'] == img]['Broken ooid'].values[0]
    print(f"  {img}: {n} broken ooids")

## 4. Visualize Sample Images with Annotations

In [ ]:
def visualize_image_with_annotations(image_name, annotations_dict, data_dir, max_display_size=1200):
    """
    Display an image with polygon overlays colored by class.
    """
    # Load image
    json_path = data_dir / f"{image_name}.json"
    json_data = load_labelme_json(json_path)
    image = load_image_from_labelme(json_data, data_dir)
    
    # Resize if too large
    h, w = image.shape[:2]
    if max(h, w) > max_display_size:
        scale = max_display_size / max(h, w)
        image = cv2.resize(image, None, fx=scale, fy=scale)
    else:
        scale = 1.0
    
    # Create overlay
    overlay = image.copy()
    
    # Color map for classes
    class_colors = {
        'Peloid': (100, 200, 100),      # Green
        'Ooid': (255, 150, 50),         # Orange
        'Broken ooid': (255, 50, 50),   # Red
        'Intraclast': (100, 150, 255),  # Blue
    }
    
    grains = annotations_dict[image_name]
    
    for grain in grains:
        points = np.array(grain['points'], dtype=np.int32)
        if scale != 1.0:
            points = (points * scale).astype(np.int32)
        
        color = class_colors.get(grain['label'], (255, 255, 255))
        
        # Draw polygon
        cv2.polylines(overlay, [points], isClosed=True, color=color, thickness=2)
        
        # Draw centroid
        cx, cy = grain['centroid']
        if scale != 1.0:
            cx, cy = int(cx * scale), int(cy * scale)
        cv2.circle(overlay, (cx, cy), 3, color, -1)
    
    # Plot
    fig, ax = plt.subplots(1, 1, figsize=(14, 10))
    ax.imshow(overlay)
    ax.axis('off')
    ax.set_title(f"{image_name} - {len(grains)} grains", fontsize=14)
    
    # Legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=np.array(color)/255, label=cls) 
                      for cls, color in class_colors.items()]
    ax.legend(handles=legend_elements, loc='upper right')
    
    plt.tight_layout()
    plt.show()

# Select a few representative images
sample_images = [
    'WT13-ES0026',  # Large image with many grains
    'WL5_IM10041',  # Contains broken ooids
    'WT15-ES0042',  # Diverse classes
]

for img_name in sample_images:
    if img_name in annotations:
        visualize_image_with_annotations(img_name, annotations, DATA_DIR)

## 5. Extract and Visualize Sample Grains

In [ ]:
def extract_grain_patch(
    image: np.ndarray,
    grain_info: dict,
    patch_size: int = 96,
    with_mask: bool = False
) -> np.ndarray:
    """
    Extract a patch centered on grain centroid.
    
    Args:
        image: Full image (H, W, 3)
        grain_info: Grain dictionary with 'centroid' and 'points'
        patch_size: Size of square patch
        with_mask: If True, multiply by mask (zeros outside grain)
        
    Returns:
        Patch of size (patch_size, patch_size, 3)
    """
    cx, cy = grain_info['centroid']
    h, w = image.shape[:2]
    
    # Calculate crop bounds
    half_size = patch_size // 2
    x1 = max(0, cx - half_size)
    y1 = max(0, cy - half_size)
    x2 = min(w, cx + half_size)
    y2 = min(h, cy + half_size)
    
    # Crop patch
    patch = image[y1:y2, x1:x2].copy()
    
    # Apply mask if requested
    if with_mask:
        # Create mask for this grain
        mask = polygon_to_mask(grain_info['points'], image.shape[:2])
        mask_crop = mask[y1:y2, x1:x2]
        
        # Multiply patch by mask (3 channels)
        patch = patch * mask_crop[:, :, np.newaxis]
    
    # Pad if necessary (near edges)
    if patch.shape[0] < patch_size or patch.shape[1] < patch_size:
        pad_h = max(0, patch_size - patch.shape[0])
        pad_w = max(0, patch_size - patch.shape[1])
        patch = np.pad(patch, ((0, pad_h), (0, pad_w), (0, 0)), mode='constant')
    
    return patch


def visualize_class_samples(annotations_dict, data_dir, n_samples=6, patch_size=96, with_mask=True):
    """
    Show sample grains from each class.
    """
    # Collect grains by class
    grains_by_class = {cls: [] for cls in VALID_CLASSES}
    
    for img_name, grains in annotations_dict.items():
        json_path = data_dir / f"{img_name}.json"
        json_data = load_labelme_json(json_path)
        image = load_image_from_labelme(json_data, data_dir)
        
        for grain in grains:
            grains_by_class[grain['label']].append((image, grain))
    
    # Plot samples for each class
    for cls in VALID_CLASSES:
        available = grains_by_class[cls]
        n_show = min(n_samples, len(available))
        
        if n_show == 0:
            print(f"No samples for {cls}")
            continue
        
        # Sample randomly
        indices = np.random.choice(len(available), n_show, replace=False)
        samples = [available[i] for i in indices]
        
        # Create subplot
        fig, axes = plt.subplots(1, n_show, figsize=(n_show * 2.5, 3))
        if n_show == 1:
            axes = [axes]
        
        fig.suptitle(f"{cls} - {len(available)} total samples", fontsize=14, fontweight='bold')
        
        for ax, (image, grain) in zip(axes, samples):
            patch = extract_grain_patch(image, grain, patch_size, with_mask)
            ax.imshow(patch)
            ax.axis('off')
        
        plt.tight_layout()
        plt.show()

# Visualize samples (with masking)
print("Sample grains from each class (with zero-masking):")
visualize_class_samples(annotations, DATA_DIR, n_samples=8, patch_size=96, with_mask=True)

## 6. Analyze Image Properties for 5-Fold CV

In [ ]:
# Calculate diversity score for each image
# Images with broken ooids are most critical

df_images['Has_Broken_Ooid'] = df_images['Broken ooid'] > 0
df_images['Has_Intraclast'] = df_images['Intraclast'] > 0
df_images['N_Classes'] = (df_images[VALID_CLASSES] > 0).sum(axis=1)

# Tier assignment for stratified splitting
def assign_tier(row):
    if row['Has_Broken_Ooid']:
        return 'Tier1_Broken_Ooid'
    elif row['Has_Intraclast']:
        return 'Tier2_Intraclast'
    else:
        return 'Tier3_Basic'

df_images['Tier'] = df_images.apply(assign_tier, axis=1)

print("Image stratification for 5-fold CV:")
print(f"\nTier distribution:")
print(df_images['Tier'].value_counts())
print(f"\nDetailed breakdown:")
display(df_images[['Image', 'Total', 'Peloid', 'Ooid', 'Intraclast', 'Broken ooid', 'N_Classes', 'Tier']])

# Recommendation
n_tier1 = (df_images['Tier'] == 'Tier1_Broken_Ooid').sum()
print(f"\n✅ Recommendation for 5-fold CV:")
print(f"  - Tier 1 images: {n_tier1} (MUST be distributed across all folds)")
print(f"  - Each fold should have at least 2 Tier 1 images")
print(f"  - Use stratified GroupKFold to ensure broken ooids in all folds")

## 7. Summary Statistics

In [ ]:
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"\nTotal images: {len(annotations)}")
print(f"Total grains (after filtering): {total_grains}")
print(f"\nClass distribution:")
for cls in VALID_CLASSES:
    count = class_counts[cls]
    pct = count / total_grains * 100
    print(f"  {cls:20s}: {count:5d} ({pct:5.2f}%)")

print(f"\nImbalance ratio (Peloid / Broken ooid): {class_counts['Peloid'] / class_counts['Broken ooid']:.1f}:1")

print(f"\nImages per tier:")
for tier in ['Tier1_Broken_Ooid', 'Tier2_Intraclast', 'Tier3_Basic']:
    n = (df_images['Tier'] == tier).sum()
    print(f"  {tier:25s}: {n} images")

print(f"\n✅ Ready for 5-fold cross-validation with stratified splitting")
print("=" * 60)